# Estudio longitudinal y análisis clásico

Este notebook organiza la reconstrucción, la auditoría y la inferencia. El texto
expone únicamente la lógica y las expresiones matemáticas. Toda observación,
fecha, cantidad, diagnóstico y resultado se obtiene durante la ejecución y se
presenta como `DataFrame` o figura.


## Entorno reproducible

La raíz del proyecto debe contener el directorio canónico `data/` y el paquete importable. El
notebook no lee tablas de resultados previamente generadas y no utiliza el
manuscrito como fuente de valores observados.


In [ ]:
from __future__ import annotations

from pathlib import Path

from IPython.display import display  # type: ignore

from festuca_analysis.longitudinal import LongitudinalNotebook

project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    raise FileNotFoundError("Ejecute el notebook desde la raíz del proyecto Festuca.")

analysis = LongitudinalNotebook(
    project_root=project_root,
    bootstrap_replicates=199,
    export_results=True,
    export_figures=True,
    figure_profile="thesis",
    print_figure_json=False,
)

## Fuente, procedencia y linaje

Los CSV normalizados de `data/` son la fuente exclusiva de las observaciones. Para cada variable se
distinguen cuatro niveles relevantes:

1. medición o identificador registrado;
2. cantidad derivada materializada en un CSV calculado;
3. estimación explícitamente identificada y separada;
4. transformación calculada por este análisis.

Las fórmulas semánticas de las variables calculadas se declaran en JSON. Las variables
analíticas se reconstruyen desde las mediciones primitivas cuando la identidad
matemática está determinada.


In [ ]:
configuration = analysis.configuration()
qa = analysis.load_data()
provenance = analysis.source_provenance()
source_audit = analysis.source_audit()
variable_lineage = analysis.variable_lineage()

assert analysis.data is not None
longitudinal_data = analysis.data.longitudinal.copy()
harvest_data = analysis.data.harvest.copy()
seed_weight_data = analysis.data.seed_weight_long.copy()

display(longitudinal_data)
display(harvest_data)
display(seed_weight_data)

## Reconstrucción determinista

Sea $m_f$ la masa fresca de la submuestra, $m_d$ su masa seca, $M_f$ la
masa fresca recolectada, $A_b$ el área de muestreo de biomasa, $m_c$ la masa
limpia de semilla, $A_h$ el área cosechada, $n_p$ el número de panojas y
$w_r$ los pesos de las réplicas técnicas.

$$
DM = 100\frac{m_d}{m_f},
\qquad
B = M_f\frac{DM}{100}\frac{10}{A_b},
\qquad
Y = m_c\frac{10}{A_h}.
$$

$$
P = \frac{n_p}{A_h},
\qquad
W_{1000}=10\,\overline{w_r},
\qquad
\widehat{S/P}=\frac{1000m_c}{W_{1000}n_p}.
$$

Para concentración de nitrógeno $N$ y una curva crítica
$N_c(W)=aW^b$:

$$
Q_N=B\frac{N}{100},
\qquad
INN=\frac{N}{N_c(B/1000)}.
$$

Los cocientes de cosecha, eficiencia agronómica y productividad aparente del
agua son transformaciones de variables anteriores. Por ello no constituyen
respuestas independientes y se interpretan como apoyo descriptivo.


In [ ]:
dry_matter_audit = analysis.flagged_dry_matter()
baseline = analysis.baseline_summary()
schedule = analysis.schedule()
water = analysis.water_inputs()

## Inferencia por fecha en bloques completos al azar

Dentro de cada sector se usa el modelo

$$
Y_{ij}=\mu+\tau_i+\beta_j+\varepsilon_{ij},
$$

con tratamiento fijo $\tau_i$ y bloque fijo $\beta_j$. Se mantienen separadas
las preguntas sobre calendarios fertilizados y la pregunta que incorpora el
tratamiento de referencia. Las comparaciones por pares se consideran
confirmatorias únicamente después de la prueba global correspondiente. Las
familias de pruebas repetidas por fecha se auditan también con ajuste por
multiplicidad.

Una prueba significativa en una fecha y no significativa en otra no constituye
por sí sola evidencia de cambio temporal. Esa pregunta se reserva para el
modelo longitudinal.


In [ ]:
rcbd_method = analysis.rcbd_functions()
date_specific_anova = analysis.longitudinal_anova()
observed_trajectories = analysis.observed_trajectories()

## Respuestas finales y contrastes planificados

El rendimiento limpio es la respuesta primaria. Los componentes de cosecha y
las transformaciones deterministas forman capas de apoyo. Los contrastes se
expresan como combinaciones lineales de medias marginales ajustadas por bloque:

$$
L=\sum_i c_i\mu_i,
\qquad
\sum_i c_i=0.
$$

La ausencia de evidencia contra una igualdad puntual no se interpreta como
prueba de equivalencia práctica. Una afirmación de equivalencia requiere un
margen definido y un procedimiento diseñado para esa pregunta.


In [ ]:
final_outcome_hierarchy = analysis.final_outcomes()
final_anova = analysis.yield_analysis()
yield_summary = analysis.yield_overview()
yield_contrasts = analysis.yield_contrasts()
yield_components = analysis.yield_components()
seed_weight_precision = analysis.seed_weight_precision()

## Acoplamiento matemático y asociaciones exploratorias

Cuando una variable contiene algebraicamente a otra, una correlación puede
aparecer aun sin un mecanismo biológico adicional. El nulo de reconstrucción
permuta el numerador estimado y vuelve a aplicar la identidad

$$
\widehat{S/P}=\frac{\widehat S}{n_p}.
$$

Las asociaciones ajustadas se obtienen mediante una regresión completa con los
controles de tratamiento y bloque. El valor $p$ corresponde al coeficiente de
la variable explicativa dentro de ese modelo, por lo que incorpora los grados
de libertad consumidos por los controles.


In [ ]:
component_reconstruction_null = analysis.component_correlations()
correlation_audit = analysis.correlation_audit()

## Diagnósticos y análisis de sensibilidad

Los diagnósticos se usan para detectar incompatibilidades entre el modelo y los
datos, no como pruebas automáticas de validez. Las sensibilidades reconstruyen
el análisis bajo políticas explícitas para materia seca y para la observación de
calidad no medida directamente. La comparación conjunta entre sectores se
mantiene descriptiva porque el sector físico no es una unidad experimental
replicada para la condición hídrica.


In [ ]:
rcbd_diagnostics = analysis.model_diagnostics()
primary_residual_diagnostics = analysis.primary_residual_diagnostics()
dry_matter_sensitivity = analysis.dry_matter_sensitivity()
missing_n_sensitivity = analysis.missing_n_sensitivity()
sector_pattern_description = analysis.joint_sector_analysis()

## Modelo longitudinal

Para parcela $k$, tratamiento $i$, bloque $j$ y fecha $t$ se compara un modelo
aditivo con un modelo que incorpora interacción tratamiento por fecha:

$$
Z_{ijkt}=\mu+\beta_j+\tau_i+\delta_t+u_k+\varepsilon_{ijkt},
$$

$$
Z_{ijkt}=\mu+\beta_j+\tau_i+\delta_t+(\tau\delta)_{it}
+u_k+\varepsilon_{ijkt},
\qquad
u_k\sim\mathcal N(0,\sigma_u^2).
$$

La respuesta se estandariza para la optimización. En biomasa se examinan tanto
la escala original como la logarítmica; ambas representan preguntas distintas,
respectivamente diferencias absolutas y proporcionales. La prueba de
interacción se calibra mediante simulación paramétrica bajo el modelo reducido.
Los intervalos de medias y contrastes usan la covarianza asintótica de los
efectos fijos y se identifican como tales.


In [ ]:
mixed_model_tests = analysis.mixed_models()
mixed_model_estimates = analysis.mixed_estimates()
application_sampling_and_scale_sensitivity = analysis.september_sensitivity()

## Síntesis reproducible

La tabla de síntesis reúne estimandos, intervalos, decisiones de diagnóstico y
reglas de interpretación. No genera conclusiones narrativas a partir de valores
codificados. Los artefactos exportados son derivados regenerables; nunca se
reutilizan como datos de entrada.


In [ ]:
automatic_summary = analysis.automatic_summary()
figure_manifest = analysis.figure_manifest()
export_manifest = analysis.export_artifacts()